In [11]:
import glfw
from OpenGL.GL import *
import numpy as np
import math


Shaders - "attribute" para dados por vértice, "uniform" para dados globais

In [12]:
vertex_code = """
    attribute vec3 position;
    uniform mat4 mat_transformation;
    void main(){
        gl_Position = mat_transformation * vec4(position, 1.0);
    }
"""

fragment_code = """
    uniform vec4 color;
    void main(){
        gl_FragColor = color;
    }
"""

Inicializacao da Janela

In [13]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)
window = glfw.create_window(700, 700, "Dois Objetos", None, None)
glfw.make_context_current(window)

### Compilação e links dos shader

In [14]:
program  = glCreateProgram()
vertex   = glCreateShader(GL_VERTEX_SHADER)
fragment = glCreateShader(GL_FRAGMENT_SHADER)

glShaderSource(vertex, vertex_code)
glShaderSource(fragment, fragment_code)

glCompileShader(vertex)
if not glGetShaderiv(vertex, GL_COMPILE_STATUS):
    raise RuntimeError(glGetShaderInfoLog(vertex).decode())

glCompileShader(fragment)
if not glGetShaderiv(fragment, GL_COMPILE_STATUS):
    raise RuntimeError(glGetShaderInfoLog(fragment).decode())

glAttachShader(program, vertex)
glAttachShader(program, fragment)
glLinkProgram(program)
glUseProgram(program)

In [ ]:
# Definição dos vértices
#
# TRIÂNGULO: 3 vértices → índices 0, 1, 2
# QUADRADO:  6 vértices (2 triângulos) → índices 3, 4, 5, 6, 7, 8
#
# Todos juntos em um único array numpy estruturado,
# exatamente como o notebook faz com a esfera.

vertices_list = [
    # -- Triângulo (lado esquerdo da tela) --
    (-0.8, -0.4, 0.0),   # base esquerda
    (-0.4, -0.4, 0.0),   # base direita
    (-0.6,  0.4, 0.0),   # topo

    # -- Quadrado (lado direito da tela) --
    # Dois triângulos que formam um quadrado.
    # Triângulo 1:
    ( 0.2, -0.4, 0.0),   # inferior esquerdo
    ( 0.6, -0.4, 0.0),   # inferior direito
    ( 0.6,  0.4, 0.0),   # superior direito
    # Triângulo 2:
    ( 0.2, -0.4, 0.0),   # inferior esquerdo  (repetido)
    ( 0.6,  0.4, 0.0),   # superior direito   (repetido)
    ( 0.2,  0.4, 0.0),   # superior esquerdo
]

# Marcamos onde cada objeto começa e quantos vértices tem.
TRIANGULO_INICIO = 0
TRIANGULO_QTDE   = 3

QUADRADO_INICIO  = 3
QUADRADO_QTDE    = 6

total = len(vertices_list)
vertices = np.zeros(total, [("position", np.float32, 3)])
vertices["position"] = vertices_list

In [ ]:
# ─────────────────────────────────────────────────────────────
# Envio dos dados para a GPU
# ─────────────────────────────────────────────────────────────
buffer_VBO = glGenBuffers(1)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_DYNAMIC_DRAW)

stride = vertices.strides[0]
offset = ctypes.c_void_p(0)

loc_position = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc_position)
glVertexAttribPointer(loc_position, 3, GL_FLOAT, False, stride, offset)

# Pegamos as localizações das uniforms uma vez só — mais eficiente
loc_color      = glGetUniformLocation(program, "color")
loc_transform  = glGetUniformLocation(program, "mat_transformation")

In [17]:
def mat_rotacao_z(angulo):
    c, s = math.cos(angulo), math.sin(angulo)
    return np.array([
        c,  -s, 0.0, 0.0,
        s,   c, 0.0, 0.0,
        0.0, 0.0, 1.0, 0.0,
        0.0, 0.0, 0.0, 1.0,
    ], np.float32)

def mat_identidade():
    return np.eye(4, dtype=np.float32).reshape(1, 16)

In [18]:
angulo_triangulo = 0.0
angulo_quadrado  = 0.0

glEnable(GL_DEPTH_TEST)
glfw.show_window(window)

while not glfw.window_should_close(window):

    # Ângulos evoluem em velocidades diferentes — fica visualmente claro
    # que cada objeto tem sua própria transformação independente
    angulo_triangulo += 0.02
    angulo_quadrado  -= 0.01   # gira no sentido oposto, mais devagar

    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    glClearColor(0.15, 0.15, 0.2, 1.0)

    # ── Desenha o TRIÂNGULO ───────────────────────────────────
    # Passo 1: define a transformação DESTE objeto
    mat = mat_rotacao_z(angulo_triangulo).reshape(1, 16)
    glUniformMatrix4fv(loc_transform, 1, GL_TRUE, mat)

    # Passo 2: define a cor DESTE objeto
    glUniform4f(loc_color, 0.2, 0.7, 1.0, 1.0)  # azul claro

    # Passo 3: desenha SOMENTE os vértices do triângulo
    # O segundo parâmetro é o índice inicial, o terceiro é a quantidade
    glDrawArrays(GL_TRIANGLES, TRIANGULO_INICIO, TRIANGULO_QTDE)

    # ── Desenha o QUADRADO ────────────────────────────────────
    # Mudamos as uniforms antes do próximo draw — a GPU vai usar
    # esses novos valores para todos os vértices do quadrado
    mat = mat_rotacao_z(angulo_quadrado).reshape(1, 16)
    glUniformMatrix4fv(loc_transform, 1, GL_TRUE, mat)

    glUniform4f(loc_color, 1.0, 0.5, 0.2, 1.0)  # laranja

    glDrawArrays(GL_TRIANGLES, QUADRADO_INICIO, QUADRADO_QTDE)

    glfw.swap_buffers(window)
    glfw.poll_events()

glfw.terminate()